# Genie Space Monitoring Solution

This notebook demonstrates how to build a comprehensive monitoring solution for AI/BI Genie spaces.
Since there is no dedicated Genie system table (yet), we combine **three data sources**:

1. **`system.access.audit`** - Audit logs filtered for `aibiGenie` service (metadata: who, when, what action)
2. **Genie Conversations API** - Actual question text, answer content, and SQL generated
3. **Combined Delta Table** - Persisted to a custom table for long-term analytics

## Architecture
```
system.access.audit ──┐
  (timestamps,        │
   user identity,     ├──► Delta Table ──► Dashboard / Alerts
   feedback events)   │    (genie_monitoring.conversation_log)
                      │
Genie Conversations ──┘
  API (question text,
   SQL generated,
   response content)
```

## Step 1: Configuration

In [0]:
# Configuration — update the default values below or override via job parameters
dbutils.widgets.text("catalog", "<your-catalog>")
dbutils.widgets.text("monitoring_schema", "genie_monitoring")
dbutils.widgets.text("genie_space_id", "<your-genie-space-id>")

CATALOG = dbutils.widgets.get("catalog")
MONITORING_SCHEMA = dbutils.widgets.get("monitoring_schema")
GENIE_SPACE_ID = dbutils.widgets.get("genie_space_id")

# Create monitoring schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{MONITORING_SCHEMA}")

print(f"Genie Space ID: {GENIE_SPACE_ID}")
print(f"Monitoring tables will be created in: {CATALOG}.{MONITORING_SCHEMA}")

## Step 2: Explore Audit Logs for Genie Events

The `system.access.audit` table captures all Genie interactions when filtered by `service_name = 'aibiGenie'`.
This gives us metadata about every interaction: who asked, when, what type of action, and feedback signals.

In [0]:
%sql
    
-- Overview of Genie audit event types
SELECT
  action_name,
  COUNT(*) AS event_count,
  COUNT(DISTINCT request_params.user_id) AS unique_users,
  MIN(event_date) AS first_seen,
  MAX(event_date) AS last_seen
FROM system.access.audit
WHERE service_name = 'aibiGenie'
  AND event_date >= CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY action_name
ORDER BY event_count DESC

In [0]:
%sql
    
-- Genie usage by user (last 7 days)
SELECT
  user_identity.email AS user_email,
  COUNT(*) AS total_events,
  COUNT(CASE WHEN action_name = 'genieStartConversationMessage' THEN 1 END) AS questions_asked,
  COUNT(CASE WHEN action_name = 'genieSendMessageFeedback' THEN 1 END) AS feedback_given,
  MIN(event_date) AS first_activity,
  MAX(event_date) AS last_activity
FROM system.access.audit
WHERE service_name = 'aibiGenie'
  AND event_date >= CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY user_identity.email
ORDER BY questions_asked DESC

In [0]:
%sql
    
-- Daily Genie usage trend
SELECT
  event_date,
  COUNT(CASE WHEN action_name IN ('genieListConversations', 'genieStartConversationMessage') THEN 1 END) AS questions,
  COUNT(CASE WHEN action_name = 'genieSendMessageFeedback' THEN 1 END) AS feedback_events,
  COUNT(DISTINCT user_identity.email) AS active_users
FROM system.access.audit
WHERE service_name = 'aibiGenie'
  AND event_date >= CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY event_date
ORDER BY event_date

In [0]:
%sql
-- Audit log events for our specific Genie space
SELECT
  event_date,
  event_time,
  action_name,
  user_identity.email AS user_email,
  request_params,
  response.status_code
FROM system.access.audit
WHERE service_name = 'aibiGenie'
  AND event_date >= CURRENT_DATE() - INTERVAL 7 DAYS
  AND request_params['space_id'] = :genie_space_id
ORDER BY event_time DESC

## Step 3: Extract Conversation Details via API

The audit logs give us **metadata** (who, when, what action) but **NOT** the actual question text or answer.
For that, we need the **Genie Conversations API**.

In [0]:
import requests
import json
from datetime import datetime

# Get workspace host and token from Databricks context
host = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

def get_all_conversations(space_id, include_all=True, page_size=100):
    """Retrieve all conversations from a Genie space, with pagination.
    
    Args:
        space_id: The Genie space ID
        include_all: If True, include conversations from all users (requires CAN MANAGE)
        page_size: Number of conversations per page (max 100)
    """
    all_conversations = []
    page_token = None

    while True:
        params = {
            "page_size": page_size,
            "include_all": str(include_all).lower()
        }
        if page_token:
            params["page_token"] = page_token

        url = f"https://{host}/api/2.0/genie/spaces/{space_id}/conversations"
        resp = requests.get(url, headers=headers, params=params)
        resp.raise_for_status()
        data = resp.json()

        all_conversations.extend(data.get("conversations", []))

        page_token = data.get("next_page_token")
        if not page_token:
            break

    return all_conversations

def get_conversation_messages(space_id, conversation_id):
    """Retrieve all messages in a conversation."""
    url = f"https://{host}/api/2.0/genie/spaces/{space_id}/conversations/{conversation_id}/messages"
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    return resp.json()

# Fetch all conversations (across all users)
print(f"Fetching conversations for space: {GENIE_SPACE_ID}")
conversations = get_all_conversations(GENIE_SPACE_ID, include_all=True)
print(f"Found {len(conversations)} conversations")

In [0]:
# Extract detailed message data from each conversation
all_messages = []

for conv in conversations:
    conv_id = conv.get("conversation_id")
    try:
        conv_detail = get_conversation_messages(GENIE_SPACE_ID, conv_id)
        messages = conv_detail.get("messages", [])

        for msg in messages:
            # Each message is a user question with Genie's response in attachments
            content = msg.get("content", "")
            msg_id = msg.get("message_id", "")
            created_ts = msg.get("created_timestamp", 0)
            last_updated_ts = msg.get("last_updated_timestamp", 0)
            status = msg.get("status", "")
            user_id = msg.get("user_id", "")

            # Extract SQL and response from attachments
            sql_query = None
            response_text = None
            query_description = None

            for att in msg.get("attachments", []):
                if att.get("query"):
                    sql_query = att["query"].get("query", "")
                    query_description = att["query"].get("description", "")
                if att.get("text"):
                    response_text = att["text"].get("content", "")

            all_messages.append({
                "conversation_id": conv_id,
                "message_id": msg_id,
                "created_timestamp": created_ts,
                "last_updated_timestamp": last_updated_ts,
                "user_id": str(user_id),
                "status": status,
                "content": content,
                "sql_query": sql_query,
                "response_text": response_text,
                "query_description": query_description
            })
    except Exception as e:
        print(f"  Error fetching conversation {conv_id}: {e}")

print(f"Extracted {len(all_messages)} messages from {len(conversations)} conversations")

In [0]:
# Convert to DataFrame for analysis
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import col, from_unixtime, to_timestamp

schema = StructType([
    StructField("conversation_id", StringType()),
    StructField("message_id", StringType()),
    StructField("created_timestamp", LongType()),
    StructField("last_updated_timestamp", LongType()),
    StructField("user_id", StringType()),
    StructField("status", StringType()),
    StructField("content", StringType()),
    StructField("sql_query", StringType()),
    StructField("response_text", StringType()),
    StructField("query_description", StringType())
])

messages_df = spark.createDataFrame(all_messages, schema=schema)
messages_df = messages_df.withColumn(
    "question_time",
    to_timestamp(from_unixtime(col("created_timestamp") / 1000))
).withColumn(
    "response_time",
    to_timestamp(from_unixtime(col("last_updated_timestamp") / 1000))
)

display(messages_df.select(
    "conversation_id", "question_time", "content", "status", "sql_query"
).orderBy("question_time"))

## Step 4: Build the Combined Monitoring Table

We now merge audit log data with conversation API data into a single, queryable Delta table.

In [0]:
# Save raw conversation data to a staging table
messages_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG}.{MONITORING_SCHEMA}.raw_conversation_messages"
)
print(f"Saved {messages_df.count()} messages to {CATALOG}.{MONITORING_SCHEMA}.raw_conversation_messages")

In [0]:
%sql
-- Create the main monitoring table combining audit logs + conversation data
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :monitoring_schema || '.conversation_log') AS

WITH conversations AS (
  -- Each row already contains the user question + Genie response (SQL, text)
  SELECT
    conversation_id,
    message_id,
    question_time,
    response_time,
    content AS question_text,
    status AS question_status,
    sql_query AS generated_sql,
    response_text,
    query_description,
    user_id,
    CASE
      WHEN response_time IS NOT NULL AND question_time IS NOT NULL
      THEN TIMESTAMPDIFF(SECOND, question_time, response_time)
    END AS response_time_seconds
  FROM IDENTIFIER(:catalog || '.' || :monitoring_schema || '.raw_conversation_messages')
  WHERE content IS NOT NULL
    AND content != ''
),

audit_users_direct AS (
  -- Match via createConversationMessage (has conversation_id)
  SELECT DISTINCT
    request_params['conversation_id'] AS conversation_id,
    user_identity.email AS user_email
  FROM system.access.audit
  WHERE service_name = 'aibiGenie'
    AND action_name = 'createConversationMessage'
    AND event_date >= CURRENT_DATE() - INTERVAL 7 DAYS
    AND request_params['space_id'] = :genie_space_id
),

audit_users_by_time AS (
  -- Match via genieStartConversationMessage by timestamp proximity
  -- This action lacks conversation_id, so we correlate by event_time ≈ question_time
  SELECT
    event_time AS audit_event_time,
    user_identity.email AS user_email
  FROM system.access.audit
  WHERE service_name = 'aibiGenie'
    AND action_name = 'genieStartConversationMessage'
    AND event_date >= CURRENT_DATE() - INTERVAL 7 DAYS
    AND request_params['space_id'] = :genie_space_id
),

audit_feedback AS (
  -- Get feedback events from audit logs
  SELECT
    request_params['conversation_id'] AS conversation_id,
    request_params['message_id'] AS message_id,
    request_params['rating'] AS feedback_rating,
    event_time AS feedback_time,
    user_identity.email AS feedback_user
  FROM system.access.audit
  WHERE service_name = 'aibiGenie'
    AND action_name = 'genieSendMessageFeedback'
    AND event_date >= CURRENT_DATE() - INTERVAL 7 DAYS
)

SELECT
  c.conversation_id,
  c.message_id,
  c.question_time,
  c.question_text,
  c.question_status,
  c.generated_sql,
  c.response_text,
  c.query_description,
  c.response_time,
  c.response_time_seconds,
  c.user_id,
  COALESCE(ud.user_email, ut.user_email) AS user_email,
  f.feedback_rating,
  f.feedback_time,
  f.feedback_user,
  CURRENT_TIMESTAMP() AS etl_timestamp
FROM conversations c
LEFT JOIN audit_users_direct ud
  ON c.conversation_id = ud.conversation_id
LEFT JOIN audit_users_by_time ut
  ON ABS(TIMESTAMPDIFF(SECOND, c.question_time, ut.audit_event_time)) <= 2
LEFT JOIN audit_feedback f
  ON c.conversation_id = f.conversation_id
  AND c.message_id = f.message_id
ORDER BY c.question_time DESC

In [0]:
%sql
-- Verify the monitoring table
SELECT * FROM IDENTIFIER(:catalog || '.' || :monitoring_schema || '.conversation_log')
ORDER BY question_time DESC

## Step 5: Monitoring Dashboards Queries

These queries can be used to create a Lakeview dashboard for ongoing Genie space monitoring.

In [0]:
%sql
-- KPI Summary: Questions asked, feedback rate, avg response time
SELECT
  COUNT(*) AS total_questions,
  COUNT(CASE WHEN generated_sql IS NOT NULL THEN 1 END) AS questions_with_sql,
  COUNT(CASE WHEN feedback_rating IS NOT NULL THEN 1 END) AS questions_with_feedback,
  ROUND(COUNT(CASE WHEN feedback_rating IS NOT NULL THEN 1 END) * 100.0 / NULLIF(COUNT(*), 0), 1) AS feedback_rate_pct,
  COUNT(CASE WHEN feedback_rating = 'POSITIVE' THEN 1 END) AS thumbs_up,
  COUNT(CASE WHEN feedback_rating = 'NEGATIVE' THEN 1 END) AS thumbs_down,
  ROUND(
    COUNT(CASE WHEN feedback_rating = 'POSITIVE' THEN 1 END) * 100.0 /
    NULLIF(COUNT(CASE WHEN feedback_rating IS NOT NULL THEN 1 END), 0), 1
  ) AS satisfaction_rate_pct,
  ROUND(AVG(response_time_seconds), 1) AS avg_response_time_sec
FROM IDENTIFIER(:catalog || '.' || :monitoring_schema || '.conversation_log')

In [0]:
%sql
-- Question frequency by hour of day
SELECT
  HOUR(question_time) AS hour_of_day,
  COUNT(*) AS question_count
FROM IDENTIFIER(:catalog || '.' || :monitoring_schema || '.conversation_log')
GROUP BY HOUR(question_time)
ORDER BY hour_of_day

In [0]:
%sql
-- Most common question patterns (first 50 chars)
SELECT
  LEFT(question_text, 50) AS question_pattern,
  COUNT(*) AS ask_count,
  COUNT(CASE WHEN feedback_rating = 'POSITIVE' THEN 1 END) AS positive_feedback,
  COUNT(CASE WHEN feedback_rating = 'NEGATIVE' THEN 1 END) AS negative_feedback
FROM IDENTIFIER(:catalog || '.' || :monitoring_schema || '.conversation_log')
GROUP BY LEFT(question_text, 50)
ORDER BY ask_count DESC

In [0]:
%sql
-- Questions that received negative feedback (improvement opportunities)
SELECT
  question_text,
  generated_sql,
  response_text,
  feedback_rating,
  question_time
FROM IDENTIFIER(:catalog || '.' || :monitoring_schema || '.conversation_log')
WHERE feedback_rating = 'NEGATIVE'
ORDER BY question_time DESC

In [0]:
%sql
-- Questions without SQL generation (Genie couldn't generate a query)
SELECT
  question_text,
  question_status,
  response_text,
  question_time
FROM IDENTIFIER(:catalog || '.' || :monitoring_schema || '.conversation_log')
WHERE generated_sql IS NULL
  AND question_status != 'COMPLETED'
ORDER BY question_time DESC

## Step 6: Schedule for Automated Refresh

To keep the monitoring table up to date, schedule this notebook as a Databricks Job:

1. **Create a Job** that runs this notebook on a schedule (e.g., every hour or daily)
2. The notebook will:
   - Pull latest conversations from the Genie API
   - Merge with audit log feedback data
   - Update the `conversation_log` table

### Incremental Refresh (Production Pattern)
For production use, implement incremental loading:
- Track the last processed conversation timestamp
- Only fetch new conversations since the last run
- Use `MERGE INTO` instead of `CREATE OR REPLACE` for the monitoring table

## Summary

| Data Source | What It Provides | Limitations |
|------------|-----------------|-------------|
| `system.access.audit` | Timestamps, user identity, feedback events, action types | No question text or answer content |
| Genie Conversations API | Question text, generated SQL, response content | Requires CAN MANAGE permission, ~28 day history |
| Combined Delta Table | Everything above, persisted long-term | Requires scheduled refresh |

### Key Monitoring Metrics
- **Adoption**: Questions per day, unique users, conversation count
- **Quality**: Feedback rate, satisfaction rate (thumbs up %), SQL generation rate
- **Performance**: Average response time
- **Improvement Areas**: Questions with negative feedback, failed queries